# Importing the packages and data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from sklearn.metrics import r2_score, mean_squared_error

from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from texttable import Texttable
import latextable

In [3]:
import sys
sys.path.insert(1, '../sar_dirichlet')
import dirichlet_regression

In [4]:
from func_test import cos_similarity, rmse_aitchison

# Loading Dataset

In [5]:
arctic = pd.read_csv('Data Dirichlet/ArcticLake.csv')

In [6]:
Y_arctic = arctic.iloc[:,:3]
X_arctic = arctic.iloc[:,3]

In [7]:
Y_arctic = np.array(Y_arctic)

In [8]:
X_arctic = np.array([[j] for j in X_arctic])

In [9]:
Z_arctic = np.ones(len(X_arctic)).reshape((39,1))

In [10]:
n_features = 1
n_classes = 3

In [11]:
X = np.array(X_arctic)
Y = np.array(Y_arctic)

X = StandardScaler().fit(X).transform(X)

In [12]:
Z = Z_arctic
gamma_0 = [0.]

In [13]:
n,K = X.shape
J = Y.shape[1]

# Order 1

In [14]:
distance_matrix = scipy.spatial.distance_matrix(X_arctic,X_arctic)
W_arctic_cont = np.zeros(np.shape(distance_matrix))
#W_arctic_cont[distance_matrix < 28] = 1
W_arctic_cont[distance_matrix < 20] = 1
# replace the 1 on the diagonal by 0
np.fill_diagonal(W_arctic_cont,0)
# scaling the matrix, so that the sum of each row is 1
W_arctic_cont = W_arctic_cont/W_arctic_cont.sum(axis=1)[:,None]

In [15]:
pd.DataFrame(W_arctic_cont).to_csv("Data Dirichlet/W_arctic_cont_20.csv", index=False)

In [17]:
neighbors = NearestNeighbors(n_neighbors=15).fit(X)
W_arctic_dist = neighbors.kneighbors_graph(X,mode='distance').toarray()
W_arctic_dist[W_arctic_dist>0] = 1/W_arctic_dist[W_arctic_dist>0]
W_arctic_dist = W_arctic_dist/W_arctic_dist.sum(axis=1)[:,None]

In [33]:
n_samples = 39

## Dirichlet model

In [30]:
%%time

pred_ns_1, pred_cont_1, pred_dist_1 = [], [], []

K=2
J=3

X_f = np.ones((n,K))
X_f[:,1:] = X

for i in range(39):  
    X_temp = np.delete(X,i,axis=0)
    Y_temp = np.delete(Y_arctic,i,axis=0)
    Z_temp = np.delete(Z_arctic,i,axis=0)
    
    Wss_cont = np.delete(W_arctic_cont,i,axis=0)
    Wss_cont = np.delete(Wss_cont,i,axis=1)
    Wss_cont = Wss_cont/Wss_cont.sum(axis=1)[:,None]
    
    Wss_dist = np.delete(W_arctic_dist,i,axis=0)
    Wss_dist = np.delete(Wss_dist,i,axis=1)
    Wss_dist = Wss_dist/Wss_dist.sum(axis=1)[:,None]
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=False)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp)
    beta = np.zeros((K, J))
    beta[:,1:] = dirichRegressor.beta
    mu = dirichlet_regression.compute_mu(X_f, beta)
    pred_ns_1.append(mu[i])

    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_cont)
    M_cont = np.identity(39) - dirichRegressor.rho * W_arctic_cont
    beta = np.zeros((K, J))
    beta[:,1:] = dirichRegressor.beta
    mu = dirichlet_regression.compute_mu_spatial(X_f, beta, M_cont)
    pred_cont_1.append(mu[i])
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_dist)    
    M_dist = np.identity(39) - dirichRegressor.rho * W_arctic_dist
    beta = np.zeros((K, J))
    beta[:,1:] = dirichRegressor.beta
    mu = dirichlet_regression.compute_mu_spatial(X_f, beta, M_dist)
    pred_dist_1.append(mu[i])

Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
Optimization terminated successfully.
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
Optimization terminated successfully.
CONVERGENCE: NO

In [42]:
print('Dirichlet Order 1 not spatial')
print(f"R2 adjusted: {1 - (1 - r2_score(Y_arctic, pred_ns_1)) * (n_samples-1)/(n-5):.3f}")
print(f"RMSE: {mean_squared_error(Y_arctic, pred_ns_1, squared=False):.3f}")
print(f"Cross-entropy: {1/n_samples*np.sum(Y_arctic*np.log(pred_ns_1)):.3f}")
print(f"Cos similarity: {cos_similarity(Y_arctic, pred_ns_1):.3f}")
print(f"RMSE_A: {rmse_aitchison(Y_arctic, pred_ns_1):.3f}")

Dirichlet Order 1 not spatial
R2 adjusted: 0.487
RMSE: 0.107
Cross-entropy: -0.944
Cos similarity: 0.963
RMSE_A: 0.805


In [67]:
print('Dirichlet Order 1 Contiguity')
print(f"R2 adjusted: {1 - (1 - r2_score(Y_arctic, pred_cont_1)) * (n_samples-1)/(n-6):.3f}")
print(f"RMSE: {mean_squared_error(Y_arctic, pred_cont_1, squared=False):.3f}")
print(f"Cross-entropy: {1/n_samples*np.sum(Y_arctic*np.log(pred_cont_1)):.3f}")
print(f"Cos similarity: {cos_similarity(Y_arctic, pred_cont_1):.3f}")
print(f"RMSE_A: {rmse_aitchison(Y_arctic, pred_cont_1):.3f}")

Dirichlet Order 1 Contiguity
R2 adjusted: 0.498
RMSE: 0.103
Cross-entropy: -0.939
Cos similarity: 0.965
RMSE_A: 0.785


In [44]:
print('Dirichlet Order 1 Distance')
print(f"R2 adjusted: {1 - (1 - r2_score(Y_arctic, pred_dist_1)) * (n_samples-1)/(n-6):.3f}")
print(f"RMSE: {mean_squared_error(Y_arctic, pred_dist_1, squared=False):.3f}")
print(f"Cross-entropy: {1/n_samples*np.sum(Y_arctic*np.log(pred_dist_1)):.3f}")
print(f"Cos similarity: {cos_similarity(Y_arctic, pred_dist_1):.3f}")
print(f"RMSE_A: {rmse_aitchison(Y_arctic, pred_dist_1):.3f}")

Dirichlet Order 1 Distance
R2 adjusted: 0.269
RMSE: 0.126
Cross-entropy: -0.969
Cos similarity: 0.948
RMSE_A: 0.907


In [22]:
# For the computation of the AIC, we take the full dataset

dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=False)
dirichRegressor.fit(X, Y_arctic, parametrization='alternative', gamma_0=gamma_0, Z=Z_arctic)
aic_ns = -2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_arctic)+2*5
print('AIC not spatial:', aic_ns)

dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
dirichRegressor.fit(X, Y_arctic, parametrization='alternative', gamma_0=gamma_0, Z=Z_arctic, W=W_arctic_cont)
aic_cont = -2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_arctic)+2*6
print('AIC contiguity:', aic_cont)

dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
dirichRegressor.fit(X, Y_arctic, parametrization='alternative', gamma_0=gamma_0, Z=Z_arctic, W=W_arctic_dist)
aic_dist = -2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_arctic)+2*6
print('AIC distance:', aic_dist)

Optimization terminated successfully.
AIC not spatial: -145.45619880071732
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
AIC contiguity: -148.46094834935343
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
AIC distance: -143.57411309123344


## Cross-entropy

In [45]:
%%time

pred_ns_1_ce, pred_cont_1_ce, pred_dist_1_ce = [], [], []

K=2
J=3

X_f = np.ones((n,K))
X_f[:,1:] = X

for i in range(39):  
    X_temp = np.delete(X,i,axis=0)
    Y_temp = np.delete(Y_arctic,i,axis=0)
    Z_temp = np.delete(Z_arctic,i,axis=0)
    
    Wss_cont = np.delete(W_arctic_cont,i,axis=0)
    Wss_cont = np.delete(Wss_cont,i,axis=1)
    Wss_cont = Wss_cont/Wss_cont.sum(axis=1)[:,None]
    
    Wss_dist = np.delete(W_arctic_dist,i,axis=0)
    Wss_dist = np.delete(Wss_dist,i,axis=1)
    Wss_dist = Wss_dist/Wss_dist.sum(axis=1)[:,None]
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=False)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp,  loss='crossentropy')
    beta = np.zeros((K, J))
    beta[:,1:] = dirichRegressor.beta
    mu = dirichlet_regression.compute_mu(X_f, beta)
    pred_ns_1_ce.append(mu[i])

    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_cont, loss='crossentropy')
    M_cont = np.identity(39) - dirichRegressor.rho * W_arctic_cont
    beta = np.zeros((K, J))
    beta[:,1:] = dirichRegressor.beta
    mu = dirichlet_regression.compute_mu_spatial(X_f, beta, M_cont)
    pred_cont_1_ce.append(mu[i])
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_dist, loss='crossentropy')    
    M_dist = np.identity(39) - dirichRegressor.rho * W_arctic_dist
    beta = np.zeros((K, J))
    beta[:,1:] = dirichRegressor.beta
    mu = dirichlet_regression.compute_mu_spatial(X_f, beta, M_dist)
    pred_dist_1_ce.append(mu[i])

Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization

In [68]:
print('CECM Order 1 not spatial')
print(f"R2 adjusted: {1 - (1 - r2_score(Y_arctic, pred_ns_1_ce)) * (n_samples-1)/(n-4):.3f}")
print(f"RMSE: {mean_squared_error(Y_arctic, pred_ns_1_ce, squared=False):.3f}")
print(f"Cross-entropy: {1/n_samples*np.sum(Y_arctic*np.log(pred_ns_1_ce)):.3f}")
print(f"Cos similarity: {cos_similarity(Y_arctic, pred_ns_1_ce):.3f}")
print(f"RMSE_A: {rmse_aitchison(Y_arctic, pred_ns_1_ce):.3f}")
print("====================")
print('Dirichlet Order 1 Contiguity')
print(f"R2 adjusted: {1 - (1 - r2_score(Y_arctic, pred_cont_1_ce)) * (n_samples-1)/(n-5):.3f}")
print(f"RMSE: {mean_squared_error(Y_arctic, pred_cont_1_ce, squared=False):.3f}")
print(f"Cross-entropy: {1/n_samples*np.sum(Y_arctic*np.log(pred_cont_1_ce)):.3f}")
print(f"Cos similarity: {cos_similarity(Y_arctic, pred_cont_1_ce):.3f}")
print(f"RMSE_A: {rmse_aitchison(Y_arctic, pred_cont_1_ce):.3f}")
print("====================")
print('Dirichlet Order 1 Distance')
print(f"R2 adjusted: {1 - (1 - r2_score(Y_arctic, pred_dist_1_ce)) * (n_samples-1)/(n-5):.3f}")
print(f"RMSE: {mean_squared_error(Y_arctic, pred_dist_1_ce, squared=False):.3f}")
print(f"Cross-entropy: {1/n_samples*np.sum(Y_arctic*np.log(pred_dist_1_ce)):.3f}")
print(f"Cos similarity: {cos_similarity(Y_arctic, pred_dist_1_ce):.3f}")
print(f"RMSE_A: {rmse_aitchison(Y_arctic, pred_dist_1_ce):.3f}")

CECM Order 1 not spatial
R2 adjusted: 0.547
RMSE: 0.102
Cross-entropy: -0.941
Cos similarity: 0.966
RMSE_A: 0.854
Dirichlet Order 1 Contiguity
R2 adjusted: 0.532
RMSE: 0.100
Cross-entropy: -0.937
Cos similarity: 0.967
RMSE_A: 0.813
Dirichlet Order 1 Distance
R2 adjusted: 0.502
RMSE: 0.106
Cross-entropy: -0.946
Cos similarity: 0.963
RMSE_A: 0.907


# Order 2

In [47]:
X_2 = np.ones((39,3))
X_2[:,1] = X[:,0]
X_2[:,2] = X[:,0]**2

## Dirichlet model

In [73]:
%%time

pred_ns_2, pred_cont_2, pred_dist_2 = [], [], []

K=3
J=3

X_f = np.copy(X_2)

for i in range(39):  
    X_temp = np.delete(X_2,i,axis=0)
    Y_temp = np.delete(Y_arctic,i,axis=0)
    Z_temp = np.delete(Z_arctic,i,axis=0)
    
    Wss_cont = np.delete(W_arctic_cont,i,axis=0)
    Wss_cont = np.delete(Wss_cont,i,axis=1)
    Wss_cont = Wss_cont/Wss_cont.sum(axis=1)[:,None]
    
    Wss_dist = np.delete(W_arctic_dist,i,axis=0)
    Wss_dist = np.delete(Wss_dist,i,axis=1)
    Wss_dist = Wss_dist/Wss_dist.sum(axis=1)[:,None]
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=False)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, fit_intercept=False)
    beta = np.zeros((K, J))
    beta[:,1:] = dirichRegressor.beta
    mu = dirichlet_regression.compute_mu(X_f, beta)
    pred_ns_2.append(mu[i])

    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_cont, fit_intercept=False)
    M_cont = np.identity(39) - dirichRegressor.rho * W_arctic_cont
    beta = np.zeros((K, J))
    beta[:,1:] = dirichRegressor.beta
    mu = dirichlet_regression.compute_mu_spatial(X_f, beta, M_cont)
    pred_cont_2.append(mu[i])
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_dist, fit_intercept=False)    
    M_dist = np.identity(39) - dirichRegressor.rho * W_arctic_dist
    beta = np.zeros((K, J))
    beta[:,1:] = dirichRegressor.beta
    mu = dirichlet_regression.compute_mu_spatial(X_f, beta, M_dist)
    pred_dist_2.append(mu[i])

Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
Optimization terminated successfully.
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_P

In [74]:
print('Dirichlet Order 2 not spatial')
print(f"R2 adjusted: {1 - (1 - r2_score(Y_arctic, pred_ns_2)) * (n_samples-1)/(n-7):.3f}")
print(f"RMSE: {mean_squared_error(Y_arctic, pred_ns_2, squared=False):.3f}")
print(f"Cross-entropy: {1/n_samples*np.sum(Y_arctic*np.log(pred_ns_2)):.3f}")
print(f"Cos similarity: {cos_similarity(Y_arctic, pred_ns_2):.3f}")
print(f"RMSE_A: {rmse_aitchison(Y_arctic, pred_ns_2):.3f}")
print("====================")
print('Dirichlet Order 2 Contiguity')
print(f"R2 adjusted: {1 - (1 - r2_score(Y_arctic, pred_cont_2)) * (n_samples-1)/(n-8):.3f}")
print(f"RMSE: {mean_squared_error(Y_arctic, pred_cont_2, squared=False):.3f}")
print(f"Cross-entropy: {1/n_samples*np.sum(Y_arctic*np.log(pred_cont_2)):.3f}")
print(f"Cos similarity: {cos_similarity(Y_arctic, pred_cont_2):.3f}")
print(f"RMSE_A: {rmse_aitchison(Y_arctic, pred_cont_2):.3f}")
print("====================")
print('Dirichlet Order 2 Distance')
print(f"R2 adjusted: {1 - (1 - r2_score(Y_arctic, pred_dist_2)) * (n_samples-1)/(n-8):.3f}")
print(f"RMSE: {mean_squared_error(Y_arctic, pred_dist_2, squared=False):.3f}")
print(f"Cross-entropy: {1/n_samples*np.sum(Y_arctic*np.log(pred_dist_2)):.3f}")
print(f"Cos similarity: {cos_similarity(Y_arctic, pred_dist_2):.3f}")
print(f"RMSE_A: {rmse_aitchison(Y_arctic, pred_dist_2):.3f}")

Dirichlet Order 2 not spatial
R2 adjusted: 0.562
RMSE: 0.095
Cross-entropy: -0.930
Cos similarity: 0.970
RMSE_A: 0.669
Dirichlet Order 2 Contiguity
R2 adjusted: 0.530
RMSE: 0.096
Cross-entropy: -0.931
Cos similarity: 0.969
RMSE_A: 0.675
Dirichlet Order 2 Distance
R2 adjusted: 0.487
RMSE: 0.101
Cross-entropy: -0.938
Cos similarity: 0.966
RMSE_A: 0.700


In [31]:
# For the computation of the AIC, we take the full dataset

dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=False)
dirichRegressor.fit(X_2, Y_arctic, parametrization='alternative', gamma_0=gamma_0, Z=Z_arctic)
aic_ns = -2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_arctic)+2*5
print('AIC not spatial:', aic_ns)

dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
dirichRegressor.fit(X_2, Y_arctic, parametrization='alternative', gamma_0=gamma_0, Z=Z_arctic, W=W_arctic_cont)
aic_cont = -2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_arctic)+2*6
print('AIC contiguity:', aic_cont)

dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
dirichRegressor.fit(X_2, Y_arctic, parametrization='alternative', gamma_0=gamma_0, Z=Z_arctic, W=W_arctic_dist)
aic_dist = -2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_arctic)+2*6
print('AIC distance:', aic_dist)

Optimization terminated successfully.
AIC not spatial: -172.56078606783757
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
AIC contiguity: -171.8199385214284
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
AIC distance: -170.67331096664614


## Cross-entropy

In [77]:
%%time

pred_ns_2_ce, pred_cont_2_ce, pred_dist_2_ce = [], [], []

K=3
J=3

X_f = np.copy(X_2)

for i in range(39):  
    X_temp = np.delete(X_2,i,axis=0)
    Y_temp = np.delete(Y_arctic,i,axis=0)
    Z_temp = np.delete(Z_arctic,i,axis=0)
    
    Wss_cont = np.delete(W_arctic_cont,i,axis=0)
    Wss_cont = np.delete(Wss_cont,i,axis=1)
    Wss_cont = Wss_cont/Wss_cont.sum(axis=1)[:,None]
    
    Wss_dist = np.delete(W_arctic_dist,i,axis=0)
    Wss_dist = np.delete(Wss_dist,i,axis=1)
    Wss_dist = Wss_dist/Wss_dist.sum(axis=1)[:,None]
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=False)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, fit_intercept=False, loss='crossentropy')
    beta = np.zeros((K, J))
    beta[:,1:] = dirichRegressor.beta
    mu = dirichlet_regression.compute_mu(X_f, beta)
    pred_ns_2_ce.append(mu[i])

    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_cont, fit_intercept=False, loss='crossentropy')
    M_cont = np.identity(39) - dirichRegressor.rho * W_arctic_cont
    beta = np.zeros((K, J))
    beta[:,1:] = dirichRegressor.beta
    mu = dirichlet_regression.compute_mu_spatial(X_f, beta, M_cont)
    pred_cont_2_ce.append(mu[i])
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_dist, fit_intercept=False, loss='crossentropy')    
    M_dist = np.identity(39) - dirichRegressor.rho * W_arctic_dist
    beta = np.zeros((K, J))
    beta[:,1:] = dirichRegressor.beta
    mu = dirichlet_regression.compute_mu_spatial(X_f, beta, M_dist)
    pred_dist_2_ce.append(mu[i])

Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization

In [78]:
print('CECM Order 2 not spatial')
print(f"R2 adjusted: {1 - (1 - r2_score(Y_arctic, pred_ns_2_ce)) * (n_samples-1)/(n-6):.3f}")
print(f"RMSE: {mean_squared_error(Y_arctic, pred_ns_2_ce, squared=False):.3f}")
print(f"Cross-entropy: {1/n_samples*np.sum(Y_arctic*np.log(pred_ns_2_ce)):.3f}")
print(f"Cos similarity: {cos_similarity(Y_arctic, pred_ns_2_ce):.3f}")
print(f"RMSE_A: {rmse_aitchison(Y_arctic, pred_ns_2_ce):.3f}")
print("====================")
print('Dirichlet Order 2 Contiguity')
print(f"R2 adjusted: {1 - (1 - r2_score(Y_arctic, pred_cont_2_ce)) * (n_samples-1)/(n-7):.3f}")
print(f"RMSE: {mean_squared_error(Y_arctic, pred_cont_2_ce, squared=False):.3f}")
print(f"Cross-entropy: {1/n_samples*np.sum(Y_arctic*np.log(pred_cont_2_ce)):.3f}")
print(f"Cos similarity: {cos_similarity(Y_arctic, pred_cont_2_ce):.3f}")
print(f"RMSE_A: {rmse_aitchison(Y_arctic, pred_cont_2_ce):.3f}")
print("====================")
print('Dirichlet Order 2 Distance')
print(f"R2 adjusted: {1 - (1 - r2_score(Y_arctic, pred_dist_2_ce)) * (n_samples-1)/(n-7):.3f}")
print(f"RMSE: {mean_squared_error(Y_arctic, pred_dist_2_ce, squared=False):.3f}")
print(f"Cross-entropy: {1/n_samples*np.sum(Y_arctic*np.log(pred_dist_2_ce)):.3f}")
print(f"Cos similarity: {cos_similarity(Y_arctic, pred_dist_2_ce):.3f}")
print(f"RMSE_A: {rmse_aitchison(Y_arctic, pred_dist_2_ce):.3f}")

CECM Order 2 not spatial
R2 adjusted: 0.584
RMSE: 0.094
Cross-entropy: -0.929
Cos similarity: 0.971
RMSE_A: 0.675
Dirichlet Order 2 Contiguity
R2 adjusted: 0.545
RMSE: 0.096
Cross-entropy: -0.931
Cos similarity: 0.969
RMSE_A: 0.675
Dirichlet Order 2 Distance
R2 adjusted: 0.562
RMSE: 0.096
Cross-entropy: -0.931
Cos similarity: 0.969
RMSE_A: 0.679
